Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
SPDX-License-Identifier: MIT-0

# Verify your domain and space

The platform team pre-provisioned the CloudWatch Omni domain, the space for this Region, the Dataset
integration and the access grants for both of your identities inside `workshop-agentcore-stack` (the
instance-role grant comes from the Code Editor stack) — Workshop Studio at event provisioning, or
`./scripts/self-service-deploy.sh` in your own account. The participant policy can read, update and grant,
but deliberately cannot create or delete domains, spaces or Dataset integrations: those belong to
CloudFormation. Every cell below reads the stack's outputs and confirms the resource is `ACTIVE`; the one
thing you may add from here is an access grant (Step 3b).

In [ ]:
%pip install --quiet "boto3==1.43.126" "botocore==1.43.126"
import boto3

assert tuple(int(x) for x in boto3.__version__.split(".")[:3]) >= (1, 43, 100), (
    f"boto3 too old: {boto3.__version__} (need >=1.43.100 for the cloudwatchomni client)"
)
# If the kernel imported an older botocore before the pip install above, the new
# service models are not loaded yet. Fail here with an actionable message.
services = boto3.session.Session().get_available_services()
assert "cloudwatchomni" in services and "observabilityadmin" in services, (
    "This kernel has no cloudwatchomni client. Restart the kernel (Kernel > Restart) and run this cell again."
)
print(f"boto3 {boto3.__version__} OK - cloudwatchomni and observabilityadmin clients available")

In [ ]:
import json
import os
import pathlib
import time

import boto3

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError("No AWS region configured. Set AWS_REGION or run `aws configure set region <region>`.")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# The CLI pages save identifiers as `export KEY=VALUE` lines in this file, so a
# participant can switch between the CLI and notebook paths at any step.
STATE_FILE = pathlib.Path.home() / ".workshop-omni.env"


def load_state():
    state = {}
    if STATE_FILE.exists():
        for line in STATE_FILE.read_text().splitlines():
            if line.startswith("export ") and "=" in line:
                key, value = line[len("export "):].split("=", 1)
                state[key] = value.strip()
    return state


def save_state(**values):
    state = load_state()
    state.update({k: v for k, v in values.items() if v})
    STATE_FILE.write_text("".join(f"export {k}={v}\n" for k, v in state.items()))
    return state


omni = boto3.client("cloudwatchomni", region_name=REGION)
obs_admin = boto3.client("observabilityadmin", region_name=REGION)
logs = boto3.client("logs", region_name=REGION)
cfn = boto3.client("cloudformation", region_name=REGION)
print(f"Region {REGION}, account ...{ACCOUNT_ID[-4:]}")

## Step 1: Read the pre-created role ARNs from the stack

In [ ]:
outputs = {o["OutputKey"]: o["OutputValue"]
           for o in cfn.describe_stacks(StackName="workshop-agentcore-stack")["Stacks"][0]["Outputs"]}
SPACE_ROLE_ARN = outputs["OmniSpaceAccessRoleArn"]
DATASET_ROLE_ARN = outputs["OmniDatasetIntegrationRoleArn"]
EVAL_ROLE_ARN = outputs["AgentCoreEvaluationRoleArn"]
for label, arn in (("Space operator role", SPACE_ROLE_ARN), ("Dataset integration role", DATASET_ROLE_ARN),
                   ("Evaluation role", EVAL_ROLE_ARN)):
    print(f"{label:26s} {arn}")

## Step 2: Find the domain

The domain name becomes the sign-in host, `https://<name>.cloudwatch-omni.global.app.aws`, and must be
unique across all of Omni, so it is derived from the account ID. `identityProviders=["IAM"]` means members
sign in as IAM principals; Identity Center is not configured in the workshop account.

In [ ]:
DOMAIN_ID = outputs.get("OmniDomainId")
if not DOMAIN_ID:
    raise RuntimeError(
        "workshop-agentcore-stack has no OmniDomainId output: it predates the Omni resources. "
        "Re-run ./scripts/self-service-deploy.sh (it updates the stack in place), then run this cell again."
    )
domain = omni.get_domain(domainId=DOMAIN_ID)["domain"]
print(f"domain {DOMAIN_ID}: name={domain['name']}  status={domain['status']}")
print(f"sign-in URL: https://{domain['name']}.cloudwatch-omni.global.app.aws")

## Step 3: Find the space

A space maps to one account and Region. The stack passed it the operator role Omni assumes to manage it and
the evaluation role AgentCore Evaluations assumes in Part B. The identity that creates a space receives a
service-managed Space Admin grant automatically — here that is the identity CloudFormation ran as
(`WSSystemRole` in a Workshop Studio event, the deployer in your own account), which is why your two
identities need the explicit grants Step 3b checks.

In [ ]:
SPACE_ID = outputs["OmniSpaceId"]
space = omni.get_space(spaceId=SPACE_ID)["space"]
print(f"space {SPACE_ID}: name={space['name']}  region={space['region']}  status={space['status']}")
print(f"evaluation role: {space.get('agentCoreEvaluationRoleArn')}")

## Step 3b: Check (or add) your browser identity's grant

Omni decides who may enter a space with **grants**, not IAM policies. Opening Omni from the AWS console
signs your browser in as the role the console session holds — `WSParticipantRole` in a Workshop Studio
event — a different principal from the one that created the space, so without a grant it is refused with
*This IAM role isn't a member of this space*. The stack granted it Space Admin already; the cell confirms
that and lists every grant (idempotent). In your own account `deploy-cfn.sh` granted the identity that ran
the deploy; to admit another one, set `CONSOLE_PRINCIPAL_ARN` to that role or user ARN first — Omni rejects
a grant for a principal that does not exist.

In [ ]:
from botocore.exceptions import ClientError

CONSOLE_PRINCIPAL_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/WSParticipantRole"   # or your own role / user ARN
CONSOLE_PRINCIPAL_TYPE = "IAM_ROLE"                                            # "IAM_USER" for a user ARN
who = CONSOLE_PRINCIPAL_ARN.rsplit("/", 1)[-1]

grants = omni.list_access_grants(domainId=DOMAIN_ID, spaceId=SPACE_ID).get("items", [])
if any(g["principal"].get("principalId") == CONSOLE_PRINCIPAL_ARN for g in grants):
    print(f"{who} already holds a grant on this space")
else:
    try:
        grant = omni.create_access_grant(
            domainId=DOMAIN_ID, spaceId=SPACE_ID, name="participant-space-admin",
            principal={"principalType": CONSOLE_PRINCIPAL_TYPE, "principalId": CONSOLE_PRINCIPAL_ARN},
            permission="SPACE_ADMIN",
        )["accessGrant"]
        print(f"Created grant {grant['name']} ({grant['grantType']}) for {who}")
    except ClientError as e:
        if "does not exist" in e.response["Error"]["Message"]:
            print(f"{CONSOLE_PRINCIPAL_ARN} does not exist in this account. Set CONSOLE_PRINCIPAL_ARN to the "
                  "role or user your browser uses in the console and re-run this cell.")
        else:
            raise

for g in omni.list_access_grants(domainId=DOMAIN_ID, spaceId=SPACE_ID).get("items", []):
    print(f"  {g['name']:45s} {g['permission']:12s} {g['grantType']:17s} {g['principal']['principalId']}")

## Step 4: Confirm telemetry flows into the Dataset

CloudWatch Logs forwards the account's log groups into the Dataset through the dataset integration
role. One integration per account and Region, created by the stack. On first enablement Omni backfills up to
seven days of existing logs and traces, so the Module 2 and 3 telemetry appears without re-running anything.

In [ ]:
integrations = obs_admin.list_dataset_integrations().get("DatasetIntegrationSummaries", [])
if not integrations:
    raise RuntimeError(
        "No Dataset integration in this Region: workshop-agentcore-stack predates the Omni resources. "
        "Re-run ./scripts/self-service-deploy.sh, then run this cell again."
    )
INTEGRATION_ARN = integrations[0]["Arn"]
print(f"Dataset integration: {INTEGRATION_ARN}")

cloudwatch = boto3.client("cloudwatch", region_name=REGION)
dataset = cloudwatch.get_dataset(DatasetIdentifier="default")
print(f"Dataset: {dataset['DatasetId']}  {dataset['Arn']}")

## Step 5: Save the identifiers for the next notebooks

In [ ]:
state = save_state(OMNI_DOMAIN_ID=DOMAIN_ID, OMNI_SPACE_ID=SPACE_ID, OMNI_EVAL_ROLE_ARN=EVAL_ROLE_ARN)
print(f"Saved to {STATE_FILE}: {sorted(state)}")

## Summary

| Resource | Scope | Purpose |
|----------|-------|---------|
| Domain `ws-platform-<account-id>` | Account | Sign-in URL and identity boundary |
| Space `workshop-space` | Account + Region | Where you query, dashboard and evaluate |
| Access grants `participant-space-admin`, `code-editor-space-admin` | Space | Admit `WSParticipantRole` (console) and the instance role (this kernel, the terminal, the deep link) |
| Dataset integration | Account + Region | Every log group and `aws/spans` forwarded into the Dataset |

Nothing you already run changed: log groups, Logs Insights queries, alarms and the GenAI Observability
dashboard keep working. Continue with `03-sign-in.ipynb`.